# Parameter Scan Exploration — Desktop World DAU (June 2026)

Human-mediated gradient-descent exploration of Prophet parameters in `DesktopModelConfig`,
aiming to bring the **June 2026 desktop world DAU forecast at 2026-12-15** up to match
the **April 2026 baseline**.

Workflow:
1. Pick a config (start from the suggestions in the empty-state cell).
2. Run it manually with `scripts/run_param_scan.py`. Each run writes into
   `param_scan_results/<slug>/`.
3. Reload this notebook (re-run the loader cell). The summary table and plots
   refresh automatically.
4. Use the **next-step recommendation** cell to pick the next config(s) to try.
5. Repeat until Dec 15 desktop world DAU is within tolerance of the April baseline.

Scope: **desktop, world (country='ALL'), os='ALL', DAU**. Mobile is excluded.


In [1]:
# [setup]
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root (notebook lives at repo root)
REPO_ROOT = Path.cwd()
assert (REPO_ROOT / "src" / "mozaic_daily").exists(), f"Expected to be at repo root, got {REPO_ROOT}"

# Make mozaic_daily importable so we can route reads through load_forecast()
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
from mozaic_daily.adjustments import load_forecast  # noqa: E402

# Baselines (no-Iran world DAU, country=ALL, segment os=ALL). Both are .raw.
APRIL_BASELINE_PARQUET = REPO_ROOT / "data-official/2026-04/desktop_cps0.15983_thresh32_recent13_clip0.6/mozaic_daily_forecast.2026-04-01.ld-D.raw.parquet"
JUNE_BASELINE_PARQUET  = REPO_ROOT / "data-official/2026-06/desktop_cps0.15983_thresh50_recent13_clip0.6/mozaic_daily_forecast.2026-05-17.ld-D.raw.parquet"
SCAN_RESULTS_DIR       = REPO_ROOT / "param_scan_results"

TARGET_DATE = pd.Timestamp("2026-12-15")

print(f"Repo root:           {REPO_ROOT}")
print(f"April baseline:      {APRIL_BASELINE_PARQUET}  (exists={APRIL_BASELINE_PARQUET.exists()})")
print(f"June  baseline:      {JUNE_BASELINE_PARQUET}  (exists={JUNE_BASELINE_PARQUET.exists()})")
print(f"Scan results dir:    {SCAN_RESULTS_DIR}  (exists={SCAN_RESULTS_DIR.exists()})")


/Users/brendanwells/work/mozaic-daily/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Users/brendanwells/work/mozaic-daily/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.bigquery_storage_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery_storage_v1 past that date.
  warnings.warn(message, FutureWarning)


Repo root:           /Users/brendanwells/work/mozaic-daily
April baseline:      /Users/brendanwells/work/mozaic-daily/data-official/2026-04/desktop_cps0.15983_thresh32_recent13_clip0.6/mozaic_daily_forecast.2026-04-01.ld-D.raw.parquet  (exists=True)
June  baseline:      /Users/brendanwells/work/mozaic-daily/data-official/2026-06/desktop_cps0.15983_thresh50_recent13_clip0.6/mozaic_daily_forecast.2026-05-17.ld-D.raw.parquet  (exists=True)
Scan results dir:    /Users/brendanwells/work/mozaic-daily/param_scan_results  (exists=False)


/Users/brendanwells/work/mozaic-daily/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# [helpers]
def world_desktop_dau_at(parquet_path: Path, target_date: pd.Timestamp,
                          ma_window: int = 1) -> float:
    """Return desktop world DAU at target_date for country='ALL', segment os='ALL'.

    Loads via ``load_forecast(path, require_state=[])``, so the file must be a
    ``.raw.`` parquet with a valid sidecar meta. Headwind-adjusted or
    state-unmarked files are rejected.

    Args:
        parquet_path: Path to a mozaic_daily_forecast.*.ld-D.raw.parquet (or full forecast).
        target_date: The date to extract the value at.
        ma_window: If >1, returns a trailing window average ending at target_date.

    Returns:
        DAU value (float). NaN if the date or slice is missing.
    """
    df, _meta = load_forecast(parquet_path, require_state=[])
    df = df[df["country"] == "ALL"]
    df = df[df["segment"] == '{"os": "ALL"}']
    df = df.copy()
    df["target_date"] = pd.to_datetime(df["target_date"])
    if ma_window > 1:
        end = target_date
        start = target_date - pd.Timedelta(days=ma_window - 1)
        window = df[(df["target_date"] >= start) & (df["target_date"] <= end)]
        if window.empty:
            return float("nan")
        return float(window["dau"].mean())
    point = df[df["target_date"] == target_date]
    if point.empty:
        return float("nan")
    return float(point["dau"].iloc[0])


# Slug regex matches both the *old* style (cps0.15983_thresh50_recent13_clip0.6)
# and the *new* style (cps..._thresh..._recent..._cpr..._ncp..._clip...).
SLUG_PATTERN_NEW = re.compile(
    r"^cps(?P<cps>[\d.]+)"
    r"_thresh(?P<thresh>\d+)"
    r"_recent(?P<recent>\d+)"
    r"_cpr(?P<cpr>[\d.]+)"
    r"_ncp(?P<ncp>\d+)"
    r"_clip(?P<clip>[\d.]+)$"
)
SLUG_PATTERN_OLD = re.compile(
    r"^cps(?P<cps>[\d.]+)"
    r"_thresh(?P<thresh>\d+)"
    r"_recent(?P<recent>\d+)"
    r"_clip(?P<clip>[\d.]+)$"
)


def parse_slug(slug: str) -> dict | None:
    """Parse a config slug; return dict of param fields or None if unrecognized."""
    m = SLUG_PATTERN_NEW.match(slug)
    if m:
        return {
            "changepoint_prior_scale": float(m["cps"]),
            "holiday_threshold": -int(m["thresh"]) / 1000.0,
            "recent_weeks": int(m["recent"]),
            "changepoint_range": float(m["cpr"]),
            "n_changepoints": int(m["ncp"]),
            "holiday_effect_floor": -float(m["clip"]),
        }
    m = SLUG_PATTERN_OLD.match(slug)
    if m:
        return {
            "changepoint_prior_scale": float(m["cps"]),
            "holiday_threshold": -int(m["thresh"]) / 1000.0,
            "recent_weeks": int(m["recent"]),
            "changepoint_range": None,
            "n_changepoints": None,
            "holiday_effect_floor": -float(m["clip"]),
        }
    return None


def load_scan_results(results_dir: Path, target_date: pd.Timestamp,
                      ma_window: int = 1) -> pd.DataFrame:
    """Walk results_dir, load each raw forecast parquet, return one row per slug.

    Only ``.raw.parquet`` files are picked up — the glob requires the state
    marker so a stale unmarked or adj-h file in a slug dir cannot contaminate
    the comparison. Prefers parameters.json (authoritative) for param values;
    falls back to parsing the slug if parameters.json is missing.
    """
    rows = []
    if not results_dir.exists():
        return pd.DataFrame()

    for slug_dir in sorted(p for p in results_dir.iterdir() if p.is_dir()):
        slug = slug_dir.name

        # Require .raw. marker — load_forecast(require_state=[]) enforces it below
        parquets = sorted(slug_dir.glob("mozaic_daily_forecast.*.ld-D.raw.parquet"))
        if not parquets:
            print(f"[skip] {slug}: no mozaic_daily_forecast.*.ld-D.raw.parquet")
            continue
        forecast_path = parquets[-1]  # most recent

        # Param values: prefer parameters.json
        params_path = slug_dir / "parameters.json"
        if params_path.exists():
            payload = json.loads(params_path.read_text())
            config = payload.get("config", payload)  # tolerate older shape
            params = {
                "changepoint_prior_scale": config.get("prophet_changepoint_prior_scale"),
                "recent_weeks":            config.get("prophet_recent_weeks"),
                "changepoint_range":       config.get("prophet_changepoint_range"),
                "n_changepoints":          config.get("prophet_n_changepoints"),
                "holiday_threshold":       config.get("holiday_threshold"),
                "holiday_max_radius":      config.get("holiday_max_radius"),
                "holiday_min_radius":      config.get("holiday_min_radius"),
                "holiday_effect_floor":    config.get("holiday_effect_floor"),
            }
        else:
            parsed = parse_slug(slug)
            if parsed is None:
                print(f"[skip] {slug}: unrecognized slug and no parameters.json")
                continue
            params = parsed

        dau = world_desktop_dau_at(forecast_path, target_date, ma_window=ma_window)
        rows.append({"slug": slug, "forecast_path": str(forecast_path),
                     **params, "dec15_world_dau": dau})

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index("slug").sort_values("dec15_world_dau")


In [3]:
# [baselines]
# Compute baseline values for reference lines on plots.
# Both point and 28-day trailing MA, since either could be the target metric.

baselines = {
    "april_point":  world_desktop_dau_at(APRIL_BASELINE_PARQUET, TARGET_DATE, ma_window=1),
    "april_ma28":   world_desktop_dau_at(APRIL_BASELINE_PARQUET, TARGET_DATE, ma_window=28),
    "june_point":   world_desktop_dau_at(JUNE_BASELINE_PARQUET,  TARGET_DATE, ma_window=1),
    "june_ma28":    world_desktop_dau_at(JUNE_BASELINE_PARQUET,  TARGET_DATE, ma_window=28),
}

print("Reference values at 2026-12-15 (desktop world DAU, no-Iran):")
for k, v in baselines.items():
    print(f"  {k:14s}: {v:>14,.0f}")

# Gap the scan is trying to close (point estimates):
gap_point = baselines["april_point"] - baselines["june_point"]
gap_ma28  = baselines["april_ma28"]  - baselines["june_ma28"]
print(f"\nGap to close (April − June, point): {gap_point:+,.0f}")
print(f"Gap to close (April − June, 28d MA): {gap_ma28:+,.0f}")


Reference values at 2026-12-15 (desktop world DAU, no-Iran):
  april_point   :     53,367,728
  april_ma28    :     48,389,006
  june_point    :     52,702,379
  june_ma28     :     47,996,913

Gap to close (April − June, point): +665,349
Gap to close (April − June, 28d MA): +392,093


In [4]:
# [load-results]
# Load all scan runs in param_scan_results/ and compute deltas vs baselines.
# Re-run this cell after each new param_scan run to refresh.

results = load_scan_results(SCAN_RESULTS_DIR, TARGET_DATE, ma_window=1)

if results.empty:
    print("=" * 70)
    print("No scan runs found in", SCAN_RESULTS_DIR)
    print("=" * 70)
    print("Run scripts/run_param_scan.py first. See the 'empty-state' cell below")
    print("for suggested starting configs.")
else:
    results = results.copy()
    results["delta_vs_april"] = results["dec15_world_dau"] - baselines["april_point"]
    results["delta_vs_june_baseline"] = results["dec15_world_dau"] - baselines["june_point"]
    cols = [
        "changepoint_prior_scale", "recent_weeks", "changepoint_range",
        "n_changepoints", "holiday_threshold", "holiday_effect_floor",
        "dec15_world_dau", "delta_vs_april", "delta_vs_june_baseline",
    ]
    print(f"{len(results)} run(s) loaded.\n")
    print(results[cols].to_string(float_format=lambda x: f"{x:,.4g}"))


No scan runs found in /Users/brendanwells/work/mozaic-daily/param_scan_results
Run scripts/run_param_scan.py first. See the 'empty-state' cell below
for suggested starting configs.


In [5]:
# [plots]
# Scatter Dec 15 DAU vs each varied parameter. For each parameter, only the
# runs where that parameter varies (>=2 distinct values) get a panel.
# Reference lines: April baseline (target) and June baseline (starting point).

SCAN_PARAMS = [
    "changepoint_prior_scale",
    "recent_weeks",
    "changepoint_range",
    "n_changepoints",
    "holiday_threshold",
    "holiday_effect_floor",
]


def plot_param_scan(df: pd.DataFrame, baselines: dict) -> None:
    if df.empty:
        print("No data to plot — run scripts/run_param_scan.py first.")
        return

    varied = [p for p in SCAN_PARAMS if df[p].nunique(dropna=True) >= 2]
    if not varied:
        print("Only one run loaded; no parameter varies yet. Plotting a single point.")
        varied = ["changepoint_prior_scale"]  # show *something*

    n = len(varied)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
    axes = axes[0]

    for ax, param in zip(axes, varied):
        sub = df.dropna(subset=[param, "dec15_world_dau"])
        ax.scatter(sub[param], sub["dec15_world_dau"], s=60, zorder=3)
        for slug, row in sub.iterrows():
            ax.annotate(slug, (row[param], row["dec15_world_dau"]),
                        xytext=(4, 4), textcoords="offset points",
                        fontsize=7, alpha=0.7)

        ax.axhline(baselines["april_point"], color="green", linestyle="--",
                   linewidth=1, label=f"April baseline ({baselines['april_point']:,.0f})")
        ax.axhline(baselines["june_point"], color="red", linestyle=":",
                   linewidth=1, label=f"June baseline ({baselines['june_point']:,.0f})")

        ax.set_xlabel(param)
        ax.set_ylabel("Dec 15 desktop world DAU")
        ax.set_title(f"vs {param}")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="lower right")

    fig.suptitle(
        f"Param scan: {len(df)} runs, target = April baseline ({baselines['april_point']:,.0f})",
        fontsize=11,
    )
    fig.tight_layout()
    plt.show()


plot_param_scan(results, baselines)


No data to plot — run scripts/run_param_scan.py first.


In [6]:
# [next-step-recommendation]
# Heuristic: for each parameter that varies, fit slope of dec15_world_dau vs param.
# The sign tells us which direction moves the forecast toward the April baseline.
# Recommend stepping further in that direction from the best run so far.

JUNE_DEFAULTS = {
    "changepoint_prior_scale": 0.15983,
    "recent_weeks": 13,
    "changepoint_range": 0.7,
    "n_changepoints": 25,
    "holiday_threshold": -0.05,
    "holiday_effect_floor": -0.6,
}

STEP_SIZES = {
    "changepoint_prior_scale": 0.05,   # additive step
    "recent_weeks": 4,                 # weeks
    "changepoint_range": 0.05,         # 0..1
    "n_changepoints": 10,              # count
    "holiday_threshold": 0.01,         # smaller magnitude = more permissive
    "holiday_effect_floor": 0.1,       # closer to 0 = less clipping
}


def recommend_next_configs(df: pd.DataFrame, baselines: dict,
                           max_recs: int = 3) -> list[dict]:
    if df.empty:
        print("No runs yet. See the empty-state cell below for starting configs.")
        return []

    target = baselines["april_point"]
    best_idx = (df["dec15_world_dau"] - target).abs().idxmin()
    best_row = df.loc[best_idx]
    best_gap = target - best_row["dec15_world_dau"]
    direction = +1 if best_gap > 0 else -1

    print(f"Best run so far: {best_idx}")
    print(f"  Dec 15 DAU: {best_row['dec15_world_dau']:,.0f}")
    print(f"  Gap to April: {best_gap:+,.0f}")
    print(f"  Need to move forecast {'UP' if direction > 0 else 'DOWN'}")
    print()

    # Per-param slope (only meaningful when that param varies)
    slopes = {}
    for param in SCAN_PARAMS:
        sub = df.dropna(subset=[param, "dec15_world_dau"])
        if sub[param].nunique() >= 2:
            xs = sub[param].astype(float).values
            ys = sub["dec15_world_dau"].astype(float).values
            slope = np.polyfit(xs, ys, 1)[0]
            slopes[param] = slope

    if not slopes:
        print("No parameter has varied yet — pick one knob and run two configs that")
        print("differ only on that knob to establish a slope.")
        # Suggest stepping each scannable knob from the best run in BOTH directions.
        recs = []
        for param, step in list(STEP_SIZES.items())[:max_recs]:
            base = best_row[param] if pd.notna(best_row[param]) else JUNE_DEFAULTS[param]
            recs.append({param: float(base) + step})
        return recs

    print("Estimated slopes (Δ DAU per +1 unit of param):")
    for p, s in slopes.items():
        print(f"  {p:30s}: {s:>+15,.0f}")
    print()

    # For each param: step in the direction that moves DAU toward target.
    recs = []
    for param, slope in sorted(slopes.items(), key=lambda kv: -abs(kv[1])):
        if abs(slope) < 1.0:
            continue  # essentially no sensitivity
        step_dir = +1 if (slope > 0) == (direction > 0) else -1
        step = STEP_SIZES[param] * step_dir
        base = best_row[param] if pd.notna(best_row[param]) else JUNE_DEFAULTS[param]
        new_val = float(base) + step
        recs.append({param: new_val})
        if len(recs) >= max_recs:
            break
    return recs


def format_rec_as_cli(rec: dict, base_cmd: str) -> str:
    flag_map = {
        "changepoint_prior_scale": "--changepoint-prior-scale",
        "recent_weeks":            "--recent-weeks",
        "changepoint_range":       "--changepoint-range",
        "n_changepoints":          "--n-changepoints",
        "holiday_threshold":       "--holiday-threshold",
        "holiday_max_radius":      "--holiday-max-radius",
        "holiday_min_radius":      "--holiday-min-radius",
        "holiday_effect_floor":    "--holiday-effect-floor",
    }
    flags = " ".join(f"{flag_map[k]} {v}" for k, v in rec.items())
    return f"{base_cmd} {flags}"


BASE_CMD = (
    "python scripts/run_param_scan.py "
    "--forecast-start-date 2026-05-17 "
    "--raw-cache-dir <first-completed-slug-dir>"
)

recs = recommend_next_configs(results, baselines)
if recs:
    print(f"\nRecommended next configs (top {len(recs)}):")
    for i, rec in enumerate(recs, 1):
        print(f"\n  [{i}] vary: {rec}")
        print(f"      $ {format_rec_as_cli(rec, BASE_CMD)}")



No runs yet. See the empty-state cell below for starting configs.


In [7]:
# [empty-state-starting-configs]
# Shows starting configs when no scan runs exist. Safe to leave in even
# after results accumulate — it just prints suggestions.

STARTING_CONFIGS = [
    {
        "note": "1. Match June baseline (no overrides) — sanity check that the runner reproduces June. "
                "This run will query BigQuery on first invocation; subsequent runs can reuse its raw cache.",
        "flags": "",
    },
    {
        "note": "2. Lower changepoint_prior_scale to smooth the recent trend (less aggressive downward).",
        "flags": "--changepoint-prior-scale 0.10",
    },
    {
        "note": "3. Shorten the recent-weeks window so seasonality leans on broader history.",
        "flags": "--recent-weeks 8",
    },
    {
        "note": "4. Pull changepoint_range back so trend changepoints stop earlier (less reactive to the tail).",
        "flags": "--changepoint-range 0.5",
    },
]

# After config 1 has run, the slug dir below will contain the raw cache parquets.
# Pass it as --raw-cache-dir to subsequent runs to skip BigQuery.
RAW_CACHE_HINT_DIR = (
    "param_scan_results/cps0.15983_thresh032_recent13_cpr0.7_ncp25_clip0.6"
)

BASE = "python scripts/run_param_scan.py --forecast-start-date 2026-05-17"

if results.empty:
    print("No scan runs yet. Suggested starting configs (run each as its own command):\n")
else:
    print("Reference: starting configs (already covered by your scan if listed above).\n")

for i, cfg in enumerate(STARTING_CONFIGS, 1):
    print(f"  {cfg['note']}")
    if i == 1:
        cmd = f"{BASE} {cfg['flags']}".rstrip()
    else:
        cmd = f"{BASE} --raw-cache-dir {RAW_CACHE_HINT_DIR} {cfg['flags']}".rstrip()
    print(f"    $ {cmd}\n")

print("Once config 1 is complete, every subsequent run can pass:")
print(f"  --raw-cache-dir {RAW_CACHE_HINT_DIR}")
print("to symlink the raw BQ parquets and skip the (slow) BigQuery query step.")



No scan runs yet. Suggested starting configs (run each as its own command):

  1. Match June baseline (no overrides) — sanity check that the runner reproduces June. This run will query BigQuery on first invocation; subsequent runs can reuse its raw cache.
    $ python scripts/run_param_scan.py --forecast-start-date 2026-05-17

  2. Lower changepoint_prior_scale to smooth the recent trend (less aggressive downward).
    $ python scripts/run_param_scan.py --forecast-start-date 2026-05-17 --raw-cache-dir param_scan_results/cps0.15983_thresh032_recent13_cpr0.7_ncp25_clip0.6 --changepoint-prior-scale 0.10

  3. Shorten the recent-weeks window so seasonality leans on broader history.
    $ python scripts/run_param_scan.py --forecast-start-date 2026-05-17 --raw-cache-dir param_scan_results/cps0.15983_thresh032_recent13_cpr0.7_ncp25_clip0.6 --recent-weeks 8

  4. Pull changepoint_range back so trend changepoints stop earlier (less reactive to the tail).
    $ python scripts/run_param_scan.py -